## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# 🤝 Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [20]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [22]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages

    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations

    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)

    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [23]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,

    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,

    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,

    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,

    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [24]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [25]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## ❓ Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:
The relationship between the three states is hierarchical - the Agent receives the query and clarifies with the user to create a research prompt. The prompt is passed to the Supervisor, which crafts areas to focus on and manages a team of Researchers that perform the work. Each Researcher simultaneously does the research and passes the summarized findings to the Supervisor, which passes the aggregated research back to the Agent, which uses this to craft the final report. 

Separating to use these three separate states is important for allowing each level to focus singularly on its task - i.e. Supervisor can handle messages with the researchers and aggregated findings, Agent can handle the user prompt and final report, etc. Additionally, having separate states allows for subagents to perform research in parallel with individual focus (i.e. a Researcher doesn't need the context of a different Researcher's topic with different state). Further, as this is a great deal of information across the three states, putting all of this in a single state would be far more context than is needed for the work being done and would come with the negatives of too much context (lost in the middle, latency, costs, etc).

## ❓ Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:
**Advantages**: Importing these components instead of including them in the notebook is a coding best practice of not containing too many lines of code in one file and allowing for reusability of the components. It allows for abstracting away the logic used to build each component so that the notebook is not too large. Using descriptive names, adding docs, and grouping imports by category in the notebook keeps this file easier to read while allowing us to have a high-level understanding of how these components are used in this notebook.

**Disadvantages**: By abstracting away the logic and limiting our understanding of each component to its name, docs, and import grouping, we don't have the full understanding of how each component is used in this notebook. We would be required to open multiple files to inspect how components are built to understand how they are used, which can lead to confusion when context switching.

## 🏗️ Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** - Write your analysis in a markdown cell below
# Clarify Prompt

## Purpose
The clarify prompt determines whether there is enough information to begin research or if clarifying questions need to be asked first. It takes the conversation messages and current date as context and outputs a decision, either asking a clarifying question or confirming that the research can begin. This ensures that the Supervisor and Researchers have a well-defined task before starting research.

## Techniques
1. **Structured JSON Output**: The prompt specifies a JSON schema with three keys (`need_clarification`, `question`, `verification`) and provides examples of what to return for each conditional path (i.e. clarification needed vs. not needed). This ensures a consistent response from the system and routes the workflow to the correct next step.

2. **Prevents Repeated Questions**: The prompt includes an instruction that if a clarifying question has already been asked, it should almost always not ask another one. This prevents an infinite clarification loop and ensures that the research workflow moves forward, only asking additional questions if absolutely necessary.

3. **Formatting Guidelines**: The prompt gives instructions to use markdown formatting, bullet points, and numbered lists when asking clarifying questions. This helps to make the user-facing output readable and well-structured.

## Suggestion
One improvement would be to add a few examples showing when clarification is needed and is not needed. For example, a vague query (i.e. "please research wellness") that needs clarification along with a detailed query (i.e. the sleep improvement query from this notebook) that does not. We've learned about the benefits of providing a few examples to the model, and this could help to the agent when determining whether to clarify or not.

---

# 🤝 Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run wellness research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [26]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [27]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [28]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [29]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [30]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [31]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [32]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [33]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [34]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research

    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)

    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research wellness strategies for improving sleep quality.

### Setup

We need to:
1. Set up the wellness research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [35]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [36]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,

        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,

        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,

        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,

        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls

        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,

        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Wellness Research

Now let's run the research! We'll ask the system to research evidence-based strategies for improving sleep quality.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [18]:
# Create our wellness research request
research_request = """
I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please research the best evidence-based strategies for improving sleep quality and create a comprehensive sleep improvement plan for me.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")

    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")

            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")

            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")

            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")

            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))

    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with researching evidence-based sleep improvement strategies. Based on your request, I understand you want a comprehensive sleep improvement plan that addresses your current challenges: inconsistent bedtimes (10pm-1am range), phone use in bed, and morning fatigue. I will now begin researching the latest scientific evidence on sleep hygiene, circadian rhythm optimization, and proven interventions to create a tailored sleep improvement plan for you.

Node: write_research_brief

Research Brief Generated:
I want to improve my sleep quality and need a comprehensive, evidence-based sleep improvement plan. My current sleep challenges include: going to bed at inconsistent times (ranging from 10pm to 1am), using my phone while in bed, and frequently feeling tired in the morning despite getting sleep. Please research the most effective, scientifically-proven strategies for improving sleep quality, w

# Evidence-Based Sleep Improvement Plan: A Comprehensive Guide to Better Sleep Quality

## Executive Summary and Scientific Foundation

Sleep quality significantly impacts overall health, with current data showing that 32.5% of adults fail to get sufficient sleep regularly [17]. Poor sleep contributes to cardiovascular disease, diabetes, obesity, and cancer, while also causing irritability, forgetfulness, and daytime drowsiness [17][18]. Your specific challenges—inconsistent bedtimes, phone use in bed, and morning fatigue—are interconnected issues that require a systematic, evidence-based approach to address the underlying circadian rhythm disruption and sleep architecture problems.

## Establishing Consistent Sleep Timing and Circadian Rhythm Optimization

### The Science of Sleep Timing Consistency

Research analyzing 92,340 participants across 14 countries demonstrates that later sleep timing and greater sleep variability are consistently associated with adverse health outcomes [1]. The study found that regularity in sleep patterns with consistent bedtimes and wake-up times should be encouraged, as earlier sleep timing and regular patterns are favorably associated with health [1]. Social jetlag—the misalignment between biological and social clocks—was specifically linked to poor health outcomes [1].

Circadian rhythms are inherent 24-hour cycles in the brain that regulate patterns of alertness and sleepiness, responding to light variations in our environment [2]. These rhythms coordinate sleep-wake cycles, hormone secretion, body temperature, and metabolism through melatonin and the pineal gland [2]. Disruptions in these rhythms can severely impact health, increasing the risk of chronic diseases [2].

### Implementation Strategy for Sleep Schedule Consistency

**Phase 1: Immediate Stabilization (Weeks 1-2)**
- Choose a consistent wake time that works with your schedule (ideally between 6:00-7:00 AM)
- Maintain this wake time even on weekends to prevent social jetlag
- Allow bedtime to naturally adjust based on sleepiness rather than forcing it initially
- Track your natural bedtime patterns during this phase

**Phase 2: Gradual Bedtime Adjustment (Weeks 3-5)**
- If your natural bedtime is later than desired, advance it by 15-30 minutes every 2-3 days
- Use the wake time as your anchor point—never allow it to shift later
- Implement light therapy protocols (detailed below) to support circadian shifting

### Light Therapy Protocol for Circadian Optimization

Morning light therapy is the most effective intervention for advancing your sleep phase earlier [7]. The timing depends entirely on which direction you need to shift your circadian rhythm: morning light immediately upon awakening at 2,500-10,000 lux for 30 minutes to 2 hours will advance your sleep phase earlier [7].

**Specific Implementation:**
- Expose yourself to bright light (5,000+ lux) within 30 minutes of your target wake time
- Use a light therapy box, or if weather permits, get natural sunlight exposure
- A single 30-minute bright-light exposure is as effective as longer durations [8]
- Continue this protocol for 3-5 weeks until your desired schedule is achieved

**Advanced Protocol: Combination Therapy**
Research shows that morning bright light combined with evening melatonin produces greater circadian phase advances than either treatment alone [9]. Consider taking 3mg of melatonin 2 hours before your desired bedtime while implementing morning light therapy, which has shown 82% improvement rates in case series [7].

## Reducing Electronic Device Impact on Sleep

### The Science of Blue Light and Sleep Disruption

Blue light exposure, particularly wavelengths between 400-500 nm, significantly disrupts circadian rhythms by affecting intrinsically photosensitive retinal ganglion cells (ipRGCs) that contain melanopsin photopigment [10]. These cells are maximally sensitive to blue light around 480 nm and send signals to the suprachiasmatic nucleus, the body's central circadian clock [10].

The research demonstrates alarming impacts of evening device use:
- Following 2-hour LED tablet exposure, students showed a 55% decrease in melatonin and 1.5-hour delayed melatonin onset compared to reading printed books [10]
- Just 2 hours of evening light exposure caused an average 1.1-hour circadian phase delay [10]
- Students using devices over 4 hours daily experience poor sleep outcomes including low efficiency and daytime dysfunction [10]

A large-scale study of 10,106 adults found that regular smartphone or tablet use was associated with sleep latency over 30 minutes (smartphones OR 1.98, tablets OR 1.44) and 1.3-1.9-fold increased risk of excessive daytime sleepiness [11].

### Evidence-Based Digital Device Management Strategy

**Immediate Implementation (Week 1):**
- Establish a complete device-free bedroom policy—remove all electronic devices from the sleeping area
- Create a charging station outside the bedroom for phones and tablets
- Use a traditional alarm clock instead of phone alarms

**Progressive Blue Light Reduction (Weeks 1-4):**
- Implement a 2-hour digital sunset: no screens 2 hours before bedtime
- If device use is necessary, use blue light blocking glasses (effective and cost less than $100) [12]
- Enable night mode/warm light settings on all devices after sunset
- Keep bedroom lighting below 100 lux to avoid melatonin suppression [12]

**Environmental Light Optimization:**
- Use red light (600-650 nanometers) for evening lighting, as red, yellow, and orange light have minimal impact on circadian rhythms [12]
- Consider red light therapy for 10-20 minutes, 30-60 minutes before bedtime, which significantly improved restorative sleep and melatonin levels in research studies [14]
- Bright bedroom lighting can decrease nocturnal melatonin production by up to 90 minutes [12]

### Alternative Evening Activities

Replace screen time with sleep-promoting activities:
- Reading physical books under warm, dim lighting
- Gentle stretching or yoga
- Meditation or breathing exercises
- Journaling or planning for the next day
- Light household tasks that don't require bright lighting

## Addressing Morning Fatigue and Optimizing Sleep Restoration

### Understanding Non-Restorative Sleep

Non-restorative sleep (NRS) can occur independently of other sleep issues and significantly impairs daytime function [15]. Research shows that individuals with morning fatigue exhibit distinct physiological differences, including decreased deep sleep duration in the first sleep cycle, decreased REM latency, and reduced delta EEG powers particularly in the first 30-65 minutes after sleep onset [16].

The key finding is that an initial decrease in heart rate variability (HRV) within the first 30 minutes of sleep may inhibit recovery from fatigue during sleep [16]. This suggests that sleep quality, not just quantity, is crucial for morning alertness.

### Sleep Architecture Optimization

**Deep Sleep Enhancement:**
Deep sleep (slow-wave sleep) is crucial for physical restoration, facilitating growth hormone release, tissue repair, immune system enhancement, and brain detoxification [3]. To optimize deep sleep:

- Maintain consistent sleep schedule (primary factor)
- Keep bedroom temperature between 65-68°F for optimal deep sleep [18]
- Avoid blue light exposure which significantly reduces deep sleep ratio [13]
- Exercise regularly (30 minutes daily, 5 days per week) but not within 3-4 hours of bedtime [19]

**Sleep Environment Optimization:**
- Create a quiet, cool, dark environment [19]
- Invest in quality mattress, pillows, and breathable bedding materials [18]
- Minimize light and noise pollution
- Consider blackout curtains or eye masks for complete darkness

### Morning Fatigue Intervention Protocol

**Immediate Wake-Up Strategy:**
- Use bright light exposure (5,000+ lux) immediately upon awakening to signal circadian awakening
- Avoid hitting snooze—fragmented sleep in the morning worsens grogginess
- Get out of bed within 5-10 minutes of waking to prevent sleep inertia

**Sleep Quality Assessment:**
If morning fatigue persists after 4-6 weeks of implementation, consider that you may have an underlying sleep disorder. Research indicates that persistent non-restorative sleep, excessive daytime sleepiness, and never feeling refreshed despite sleeping through the night suggests a primary sleep disorder rather than simple sleep hygiene issues [22]. Conditions like obstructive sleep apnea, periodic limb movement disorder, and narcolepsy must be excluded [22].

## Comprehensive Sleep Hygiene Protocol

### The Complete Evidence-Based Framework

The National Sleep Foundation recommends six key components for healthy sleep [19]:

1. **Light Exposure Management:** Bright natural light exposure during the day, controlled lighting in the evening
2. **Regular Exercise:** 30 minutes daily, 5 days per week, but not close to bedtime
3. **Consistent Meal Timing:** Regular meal schedules support circadian rhythm regulation
4. **Substance Management:** Avoid heavy meals, nicotine, caffeine, and alcohol before bedtime
5. **Wind-Down Routine:** Consistent relaxing routines targeting 7-9 hours of sleep for adults
6. **Optimal Environment:** Device-free bedroom with quiet, cool, dark conditions

### Advanced Sleep Hygiene Techniques

**The 20-Minute Rule:**
If you spend around 20 minutes in bed without falling asleep, get out of bed and do something relaxing in low light, then return when you feel tired [18]. This prevents bed anxiety and maintains the association between bed and sleep.

**Progressive Muscle Relaxation:**
Systematically tense and release muscle groups starting from your toes and working up to your head. This technique helps reduce physical tension that may be preventing quality sleep.

**Breathing Techniques:**
Implement the 4-7-8 breathing pattern: inhale for 4 counts, hold for 7 counts, exhale for 8 counts. This activates the parasympathetic nervous system and promotes relaxation.

## Implementation Timeline and Expected Outcomes

### Week 1-2: Foundation Phase
**Actions:**
- Remove all devices from bedroom
- Establish consistent wake time
- Begin morning light exposure routine
- Start tracking sleep patterns

**Expected Outcomes:**
- Initial adjustment period with possible temporary sleep disruption
- Beginning of circadian rhythm stabilization

### Week 3-4: Optimization Phase
**Actions:**
- Fine-tune bedtime consistency
- Implement complete blue light management
- Optimize sleep environment (temperature, darkness, quiet)
- Add evening wind-down routine

**Expected Outcomes:**
- Noticeable improvement in sleep latency
- Reduced middle-of-night awakenings
- Beginning improvement in morning alertness

### Week 5-8: Consolidation Phase
**Actions:**
- Maintain all established routines
- Consider melatonin supplementation if needed (consult healthcare provider)
- Assess and adjust based on sleep quality improvements

**Expected Outcomes:**
- Consolidated sleep schedule with consistent timing
- Significant improvement in morning energy levels
- Reduced daytime fatigue and improved cognitive function

### Week 9-12: Maintenance and Assessment
**Actions:**
- Continue all protocols with minor adjustments as needed
- Assess overall sleep quality and daytime functioning
- Consider professional evaluation if significant issues persist

**Expected Outcomes:**
- Stable, high-quality sleep pattern
- Optimal daytime alertness and cognitive performance
- Long-term health benefits from improved sleep

## Personalization Considerations and When to Seek Professional Help

### Individual Variation Factors

While these recommendations are broadly applicable, several factors may require personalization:
- **Age:** Older adults may need different approaches, though 7-8 hours remains optimal [20]
- **Work Schedule:** Shift workers require specialized circadian management strategies
- **Medical Conditions:** Certain medications or health conditions may affect sleep architecture
- **Chronotype:** Natural early birds vs. night owls may need different timing adjustments

### Professional Evaluation Criteria

Consider seeking professional sleep medicine evaluation if:
- Morning fatigue persists after 6-8 weeks of consistent implementation
- You experience excessive daytime sleepiness requiring stimulants
- Never feeling refreshed despite sleeping through the night
- Loud snoring, breathing interruptions, or restless leg symptoms
- Severe insomnia lasting more than 3 months

Research shows that patients with similar sleep study results often have different clinical profiles, requiring tailored diagnostic and therapeutic strategies [21]. A comprehensive sleep study including polysomnography may be necessary to rule out sleep disorders like sleep apnea, periodic limb movement disorder, or narcolepsy [22].

### Maintenance and Long-Term Success

Sleep health recommendations should be followed throughout the lifespan as a means to prevent or delay adverse effects of poor sleep on health and functioning [20]. The key to long-term success is understanding that sleep hygiene is not a temporary intervention but a permanent lifestyle modification that supports optimal health and cognitive function.

Regular assessment and minor adjustments based on life changes, seasonal variations, and aging will help maintain the benefits of your improved sleep system. Remember that healthy sleep requires adequate duration (7+ hours for adults), appropriate timing, regularity, good quality, and absence of sleep disorders [17].

### Sources

[1] Sleep timing, sleep consistency, and health in adults: a systematic review: https://pubmed.ncbi.nlm.nih.gov/33054339/
[2] Exploring the Role of Circadian Rhythms in Sleep and Recovery: https://www.cureus.com/articles/258897-exploring-the-role-of-circadian-rhythms-in-sleep-and-recovery-a-review-article
[3] Exploring the Role of Circadian Rhythms in Sleep and Recovery: https://pmc.ncbi.nlm.nih.gov/articles/PMC11221196/
[4] Circadian Rhythm Sleep-Wake Disorders: https://www.sleephealth.org/circadian-rhythm-sleep-wake-disorders/
[5] Clinical Practice Guideline for the Treatment of Intrinsic: https://aasm.org/resources/clinicalguidelines/crswd-intrinsic.pdf
[6] Circadian Rhythm Treatment Guidelines Updated: https://aastweb.org/circadian-rhythm-treatment-guidelines-updated/
[7] What is the optimal timing and dose of bright light therapy: https://www.droracle.ai/articles/612529/what-is-the-optimal-timing-and-dose-of-bright
[8] Phase advancing human circadian rhythms with morning: https://www.sciencedirect.com/science/article/abs/pii/S1389945714004936
[9] Combining bright light with melatonin produces greater advance in circadian timing: https://aasm.org/combining-bright-light-with-melatonin-produces-greater-advance-in-circadian-timing/
[10] Impacts of Blue Light Exposure From Electronic Devices on: https://www.chronobiologyinmedicine.org/journal/view.php?number=167&viewtype=pubreader
[11] The impact of bedtime technology use on sleep quality and: https://pmc.ncbi.nlm.nih.gov/articles/PMC8906383/
[12] How Electronics Affect Sleep - Sleep Foundation: https://www.sleepfoundation.org/how-sleep-works/how-electronics-affect-sleep
[13] Effects of pre-bedtime blue-light exposure on ratio of deep sleep in: https://www.sciencedirect.com/science/article/abs/pii/S1389945721003257
[14] Red Light Therapy for Sleep: Does It Work?: https://sleep.me/post/red-light-therapy-for-sleep?srsltid=AfmBOoqLY7OH5cgIoWmDqljtiSQ0gArWxe73muwPuvJKHbfjeyqZydrc
[15] Nonrestorative Sleep as a Distinct Component of Insomnia: https://pmc.ncbi.nlm.nih.gov/articles/PMC2849783/
[16] Non-restorative Sleep Caused by Autonomic and: https://pmc.ncbi.nlm.nih.gov/articles/PMC6370690/
[17] Sleep is essential to health: an American Academy of: https://pmc.ncbi.nlm.nih.gov/articles/PMC8494094/
[18] How to Sleep Better: https://www.sleepfoundation.org/sleep-hygiene/healthy-sleep-tips
[19] National Sleep Foundation: https://www.thensf.org/
[20] Promoting Healthy Sleep for Older Adults: https://www.thensf.org/wp-content/uploads/2022/12/NSF-2022-OlderAdults-Report-Digital.pdf
[21] AI-driven clinical decision support for early diagnosis and: https://www.nature.com/articles/s41533-025-00455-5
[22] What is the best course of treatment for a patient with: https://www.droracle.ai/articles/716355/what-is-the-best-course-of-treatment-for-a


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your sleep issues, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., sleep hygiene, circadian rhythm, blue light effects)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive sleep improvement plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## ❓ Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:
**Advantages of Parallel Researchers over Sequential Research**: Parallel researchers are better suited for open-ended, complex tasks that require multiple areas of focus for answering a user's query. Using multiple asynchronous researchers can offer a speed advantage over a single research agent and offer comparisons between the multiple researchers' findings. Parallel researchers should be used for open ended, time-sensitive queries where the findings of each researcher is not required to build upon the findings of other researchers.

**Disadvantages**: Using multiple asynchronous agents can be overkill for simpler queries that can be answered quickly by a single agent. Additionally, using a team of multiple researchers will cost more as token usage increases, in addition to architecting a solution that keeps the parallel researchers within token limits via compression. The sequential research approach should be used when the query requires researchers to build upon the findings of other researchers or if the output should follow a pre-defined, structured output, as parallel research architecture introduces added complexity and unpredictability.

## ❓ Question #4:

How would you adapt this deep research architecture for a production wellness application? What additional components would you need?

##### Answer:
For a production wellness application, we would need to add persistent memory (via PostgresStore) to avoid losing data every time the server restarts as well as checkpointing to resume interrupted research workflows. We would likely want to add monitoring and observability (via LangSmith) to monitor how the app is performing and track any errors. Additionally, adding cost-related considerations, such as limits to usage, using cost-effective models for compression & summarization, and determining how many parallel researchers to use would help to prevent users from straining our app / costs. Further, we could add a component to ensure that our web search results are credible and accurate, as well as adding domain-specific MCP tools to use reliable sources of information and ensure that we are providing safe responses, considering the sensitive nature of wellness queries.

## 🏗️ Activity #2: Custom Wellness Research

Using what you've learned, run a custom wellness research task.

**Requirements:**
1. Create a wellness-related research question (exercise, nutrition, stress, etc.)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research exercise routines for specific conditions (bad knee, lower back pain)
- Compare different stress management techniques
- Investigate nutrition strategies for specific goals
- Explore meditation and mindfulness research

**YOUR CODE HERE**

In [40]:
# YOUR CODE HERE
# Create your own wellness research request and run it

my_wellness_request = """
I struggle with chronic lower back pain and want to find safe, effective exercise routines to help manage and reduce it. I currently:
- Sit at a desk for 8+ hours per day
- Experience stiffness and soreness in my lower back, especially in the morning and after long periods of sitting
- Have avoided exercise out of fear of making the pain worse

Please research evidence-based exercise routines, stretches, and movement strategies specifically designed for people with lower back pain. I'd like a comprehensive plan that includes what exercises to do, how often, and any exercises or movements I should avoid.
"""

# Optionally modify the config
my_config = {
    "configurable": {
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        "allow_clarification": False,             # Disabled - our query is detailed and specific
        "max_concurrent_research_units": 2,       # 2 parallel researchers for distinct sub-topics
        "max_researcher_iterations": 3,           # More iterations for thorough health research
        "max_react_tool_calls": 5,                # More tool calls per researcher for deeper searching
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

# Run your research
async def run_custom_research(request, config):
    """Run a custom research workflow and display results."""
    print("Starting custom wellness research workflow...\n")

    async for event in graph.astream(
        {"messages": [{"role": "user", "content": request}]},
        config,
        stream_mode="updates"
    ):
        for node_name, node_output in event.items():
            if node_output is None:
                continue

            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")

            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")

            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")

            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")

            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))

    print("\n" + "="*60)
    print("Custom wellness research workflow completed!")
    print("="*60)

await run_custom_research(my_wellness_request, my_config)

Starting custom wellness research workflow...


Node: write_research_brief

Research Brief Generated:
I need a comprehensive, evidence-based exercise and movement plan to help manage and reduce my chronic lower back pain. I sit at a desk for 8+ hours per day and experience stiffness and soreness in my lower back, especially in the morning and after long periods of sitting. I have avoided exercise out of fear of making the pain worse. Please research and provide: (1) Specific evidence-based exercise routines and stretches designed for people with chronic lower back pain, including detailed descri...



Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Exercise and Movement Plan for Chronic Lower Back Pain Management

## Evidence-Based Exercise Routines and Detailed Instructions

### Core Stabilization Protocol

Research demonstrates that exercise therapy is highly effective for chronic lower back pain, with rehabilitation emphasizing core strength, low back support, and posture correction resulting in significant alleviation of pain and disability [1]. A comprehensive core stabilization protocol provides the foundation for safe pain management.

**Basic Exercises for Beginners:**

- **Marching in Place**: Perform 20-40 repetitions on each side while maintaining proper posture and engaging core muscles
- **Upper Body Twist**: Hold gentle rotations for 20 seconds, keeping the lower body stable
- **Trunk Flexion Stretch**: Complete 3 repetitions with 20-second holds, gently stretching the lower back
- **Seated "Cat/Camel" Stretch**: Perform 10 repetitions in each direction, alternating between arching and rounding the spine
- **Seated Hip Stretch**: Hold for 20-60 seconds to address tight hip flexors common in desk workers [1]

**Progressive Strengthening Exercises:**

- **Abdominal Curls**: Start with 10 repetitions for 2 sets, focusing on controlled movement rather than speed
- **Oblique Strengthening**: Perform 10 repetitions for 2 sets on each side to target lateral core muscles
- **Crossover Stabilization Exercises**: Progress through phases 1-4 based on tolerance and ability
- **Stability Chair Squats**: Complete 10 repetitions for 2 sets, using a chair for support initially [1]

### Mayo Clinic Three-Tier Exercise Program

The Mayo Clinic's evidence-based program organizes exercises by progressive difficulty levels, allowing safe advancement as strength and confidence improve [5].

**Tier 1 - Flexibility Exercises:**
- **Knee-to-Chest Stretches**: Pull one knee up to your chest until a comfortable stretch is felt in the lower back and buttocks, holding for 15-30 seconds
- **Lower Trunk Rotation**: Gentle spinal twists while lying down to improve mobility
- **Hamstring Stretches**: Address tight posterior leg muscles that contribute to lower back tension
- **Piriformis Stretches**: Target hip and buttock muscles that affect lower back mechanics
- **Hip Flexor Stretches**: Essential for desk workers to counteract prolonged sitting positions [5]

**Tier 2 - Stabilization Exercises:**
- **Pelvic Tilts**: Flatten your lower back onto the floor by tightening stomach muscles, holding for 5-10 seconds
- **Partial Curl-ups**: Progress through difficulty levels while maintaining proper form
- **Prone Leg Raises**: Tighten abdominals to keep trunk rigid while slowly raising straight leg 6-8 inches from the floor
- **Standing Pelvic Tilts**: Practice core engagement in functional positions
- **Wall Slides**: Perform controlled squatting motions against a wall for support [5]

**Tier 3 - Advanced Stabilization:**
- **Simultaneous Arm and Leg Movements**: While maintaining pelvic tilt, raise opposite arm and leg simultaneously
- **Bridging with Leg Raises**: Combine bridge position with alternating straight leg raises
- **All-Fours Exercises**: Raise opposite arm and leg while maintaining neutral spine
- **Half-Kneel to Standing Transitions**: Progress to more dynamic, functional movements [5]

## Frequency, Duration, and Progression Guidelines

### Optimal Exercise Prescription Parameters

A comprehensive network meta-analysis of 26 randomized controlled trials involving 1,507 participants identified the most effective exercise prescription parameters for chronic lower back pain management [11].

**Most Effective Exercise Prescription:**
- **Duration**: 15-30 minutes per session showed superior outcomes (SMD = -1.62)
- **Frequency**: 3 sessions per week demonstrated optimal results (SMD = -1.44)
- **Duration of Program**: Programs lasting 16 weeks or longer were most effective (SMD = -2.75)
- **Exercise Type**: Tai Chi ranked highest in effectiveness (SUCRA = 77.4) due to its slow, continuous, mindful approach that helps regulate muscle tension [11]

### Evidence-Based Progression Criteria

Research reveals that exercise progression should be based on both subjective and objective measures for optimal safety and effectiveness [14].

**Subjective Progression Indicators:**
- Patient performing exercises successfully or easily
- Reduced pain levels during and after exercise
- Decreased perceived fatigue
- Improved exercise tolerance
- Enhanced confidence in movement [14]

**Objective Progression Markers:**
- Increased time duration of exercise tolerance
- Higher repetition counts achieved comfortably
- Additional sets completed without excessive fatigue
- Progressive load increases based on performance milestones
- Improved functional movement patterns [14]

### Safe Starting Guidelines

For individuals with chronic lower back pain and exercise avoidance, begin with the lowest intensity exercises and progress gradually. Start with 2-3 basic exercises from the flexibility tier, performing them every other day for the first week. Increase frequency to daily practice in week two, then gradually add stabilization exercises in weeks 3-4 based on pain response and comfort level.

## Movement Strategies and Ergonomic Recommendations for Desk Workers

### Proper Workspace Setup

Prolonged sitting adds tremendous pressure to back muscles and spinal discs, with sitting being identified as a major cause of back pain and increased stress on the back, neck, arms, and legs [7].

**Essential Ergonomic Setup Requirements:**

**Chair and Desk Configuration:**
- **Elbow Position**: Upper arms should be parallel to spine with elbows at 90-degree angle when hands rest on work surface
- **Thigh Clearance**: You should be able to slide fingers under thigh at chair's leading edge; use footrest if space is too tight
- **Calf Space**: A clenched fist should fit between the back of calf and front of chair
- **Lower-Back Support**: Buttocks should be pressed against chair back with lumbar cushion causing slight lower back arch to prevent forward slumping
- **Eye Level**: Computer screen center should be at eye level, with top of screen at eye level or 15 degrees below [7]

**Monitor and Input Device Positioning:**
- **Screen Distance**: Position monitor arm's length away (approximately 18-24 inches)
- **Keyboard Height**: Allow elbows to bend at 90 degrees with arms close to sides
- **Mouse Placement**: Keep mouse close to keyboard to avoid reaching
- **Document Holder**: Position at same level as monitor to prevent neck strain [10]

### Movement and Break Strategies

Research demonstrates that no matter how comfortable the workspace setup, prolonged static posture is detrimental to back health [7].

**Optimal Break Schedule:**
- **Micro-breaks**: Take 10-15 second breaks frequently throughout tasks - look away from monitor, stand up, or stretch arms
- **Regular Movement**: Stand, stretch, and walk for at least 1-2 minutes every 30 minutes
- **Longer Breaks**: Take 30-60 minute breaks for more substantial movement and position changes [7][10]

**Specific Movement Strategies:**
- Set phone alarms as movement reminders
- Use standing desk options when available
- Take walking meetings when possible
- Perform desk-based stretches throughout the day
- Use bathroom breaks as opportunities for brief walks [9]

### Core Strengthening Integration

A strong core provides better spinal support and helps alleviate back pain, while weak core muscles increase the risk of developing back pain [6]. Desk workers should incorporate exercises like planks into their routine as part of comprehensive back pain prevention, as these exercises can be performed in office settings or at home.

## Exercises and Movements to Avoid

### Contraindicated Exercise Types

Research identifies specific exercise approaches that lack evidence and should be avoided for chronic lower back pain management.

**Ineffective or Potentially Harmful Approaches:**
- **Back Schools**: Lack convincing evidence and cannot be recommended
- **Sensory Discrimination Training**: No proven benefit for chronic lower back pain
- **Proprioceptive Exercises**: Insufficient evidence for effectiveness
- **Sling Exercises**: Cannot be recommended based on current research
- **Pure Cardiorespiratory Exercise**: Shows no effect on reducing low back pain when used alone [8][3]

### Harmful Movement Patterns

**High-Impact Activities**: Avoid exercises involving jumping, jarring movements, or high-impact activities until core stability and pain levels improve significantly.

**Extreme Range of Motion**: Avoid exercises that push the spine into extreme flexion (excessive forward bending) or extension (excessive backward bending) positions.

**Heavy Lifting Without Proper Form**: Avoid deadlifts, heavy squats, or other loaded spinal movements until proper movement patterns are established and supervised by a healthcare provider.

**Rotational Movements Under Load**: Avoid exercises combining spinal rotation with resistance or weight until core stability is well-established.

### Clinical Contraindications

The American Physical Therapy Association guidelines specify that intermittent or static lumbar traction should not be utilized for reducing symptoms in patients with acute, subacute, or chronic low back pain [4]. Additionally, patient education approaches that directly or indirectly increase perceived threat or fear associated with low back pain should be avoided [4].

## Safe Exercise Initiation Guidelines and Warning Signs

### Understanding and Addressing Fear-Avoidance

Fear-avoidance behavior is common among individuals with chronic lower back pain, with 43% of patients reporting significant fear-avoidance beliefs about physical activity [12]. This fear can create a cycle where pain is misinterpreted in a catastrophic manner, leading individuals to exaggerate the threat of pain and avoid beneficial movement [17].

**Strategies for Overcoming Exercise Fear:**
- **Pain Neuroscience Education**: Learning about how pain works can reduce limiting beliefs about movement
- **Graded Exposure**: Systematic confrontation with feared movements to reduce avoidance behaviors
- **Behavioral Experiments**: Challenging harm expectations through controlled movement experiences
- **Cognition-Targeted Exercise**: Approaches that address maladaptive pain beliefs while engaging in movement [13][8]

### Critical Warning Signs Requiring Immediate Medical Attention

**Emergency Symptoms:**
- **Neurological Symptoms**: Leg weakness, numbness, or loss of sensation
- **Bowel or Bladder Dysfunction**: Loss of control or significant changes in function
- **Severe Progressive Weakness**: Rapidly worsening muscle weakness in legs
- **Saddle Anesthesia**: Numbness in the groin or inner thigh area [16]

**Serious Conditions Requiring Medical Evaluation:**
- Signs of infection (fever, chills with back pain)
- Symptoms suggesting compression fractures
- Indicators of abdominal aneurysm
- Signs of cauda equina syndrome
- Tumor-related symptoms [4]

### Exercise Safety Principles

**Pre-Exercise Medical Clearance**: Consult a healthcare provider before beginning any exercise program to ensure appropriate exercise instructions and safety precautions [1].

**Progressive Loading Principles**: Start with exercises that can be performed pain-free and gradually increase difficulty based on both subjective comfort and objective performance markers [14].

**Pain Monitoring Guidelines**: 
- Exercise should not significantly increase pain levels
- Some mild discomfort during movement is normal and expected
- Severe pain increases warrant immediate exercise cessation
- Post-exercise pain should return to baseline within 24 hours

**Functional Movement Emphasis**: Focus on exercises that promote normal movement patterns rather than isolated muscle strengthening alone.

### Exercise Modification Strategies for Beginners

**Pain-Sensitive Modifications:**
- Reduce range of motion to comfortable levels
- Decrease number of repetitions and sets initially
- Use supported positions (wall support, chair assistance)
- Focus on slow, controlled movements rather than speed
- Employ shorter exercise sessions with more frequent practice

**Confidence-Building Approaches:**
- Start with exercises performed in supported positions (lying down, sitting)
- Use visual feedback when possible (mirrors, video guidance)
- Keep exercise logs to track progress and build confidence
- Celebrate small improvements and milestone achievements
- Consider group exercise classes specifically designed for chronic pain management

### Long-Term Success Strategies

Research demonstrates that exercise therapy is most effective when combined with psychological and social components using a biopsychosocial approach [8]. Behavioral psychological interventions show superior long-term pain reduction compared to treatments without psychological components, particularly at long-term follow-up periods.

**Sustainable Exercise Integration:**
- Develop routines that fit into daily schedules
- Combine exercise with enjoyable activities when possible
- Build social support through exercise partners or groups
- Set realistic, achievable goals
- Plan for setbacks and develop coping strategies
- Regular reassessment and program modification based on progress

The evidence consistently shows that the era of bed rest for back pain has ended, and there is a crucial middle ground between excessive activity and complete inactivity that physical therapy and structured exercise programs can help identify [9].

### Sources

[1] Core Stabilization for Low Back Pain Protocol: https://www.danielparkmd.com/pdfs/physical-therapy-back-pain.pdf
[2] Chronic Low Back Pain: https://www.physio-pedia.com/Chronic_Low_Back_Pain
[3] Exercise interventions for the treatment of chronic low back pain: https://www.hsrd.research.va.gov/meetings/sota/pain/Exercise/ExerciseWG_Searle.pdf
[4] Low Back Pain: Clinical Practice Guidelines Linked to the International Classification of Functioning, Disability, and Health: https://pmc.ncbi.nlm.nih.gov/articles/PMC4893951/
[5] Low Back Pain Exercises - MC7245-464: https://www.mayoclinichealthsystem.org/-/media/national-files/documents/hometown-health/2021/low-back-pain-exercises.pdf
[6] Tips for Office Workers to Avoid Back Pain: https://www.isppcenter.com/blog/tips-for-office-workers-to-avoid-back-pain
[7] Ergonomic and Proper Posture for Sitting - Spine Care: https://www.uclahealth.org/medical-services/spine/patient-resources/ergonomics-prolonged-sitting
[8] Exercise and Chronic Low Back Pain: https://www.iasp-pain.org/resources/fact-sheets/exercise-and-chronic-low-back-pain/
[9] Effective Ergonomic Strategies to Prevent Back Pain in the Office: https://news.briotix.com/ergonomic-strategies-to-prevent-back-pain-office
[10] Office Ergonomics: https://healthy.kaiserpermanente.org/health-wellness/health-encyclopedia/he.office-ergonomics.tr5915
[11] Exercise prescription for improving chronic low back pain: https://www.frontiersin.org/journals/public-health/articles/10.3389/fpubh.2025.1512450/full
[12] Fear-Avoidance Beliefs for Physical Activity Among Chronic Low Back Pain Patients: https://pmc.ncbi.nlm.nih.gov/articles/PMC9885961/
[13] Unraveling the role of fear and avoidance behavior in chronic musculoskeletal pain: https://www.sciencedirect.com/science/article/abs/pii/S1413355525000267
[14] Therapeutic Exercise Progression in Patients with Nonspecific Low Back Pain: https://pmc.ncbi.nlm.nih.gov/articles/PMC12676137/
[15] Fear-avoidance beliefs increase the perception of pain and disability: https://journals.lww.com/md-journal/fulltext/2025/07250/fear_avoidance_beliefs_increase_the_perception_of.94.aspx
[16] When to get help for low back pain: https://www.health.harvard.edu/pain/when-to-get-help-for-low-back-pain
[17] Fear Avoidance Model: https://www.physio-pedia.com/Fear_Avoidance_Model


Custom wellness research workflow completed!


### Documentation

**What Worked Well**: Disabling clarification was necessary because the clarify node was asking an unnecessary question despite the detailed input - setting it to false allowed the workflow to move on to the research brief. Increasing max_concurrent_research_units to 2 allowed the Supervisor to delegate distinct sub-topics in parallel & increasing max_researcher_iterations to 3 and max_react_tool_calls to 5 gave researchers more power to search deeply towards a better response.

**What Could Be Improved**: Having to disable clarification is not ideal - in my opinion the prompt was well defined, which suggests the clarify prompt's threshold for when to ask could be improved (i.e. with few-shot examples). Additionally, disabling clarification caused the node to return None, which required adding a guard in our streaming logic that the original run_research function did not need. We could also bump max_concurrent_research_units higher if cost is not a concern, as the topic could justify more parallelism.